In [ ]:
# Cell 1 — Drive Mount & Imports

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Standard
import os
import json
import time
import random
import math
import numpy as np
from copy import deepcopy

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# HuggingFace
from datasets import load_dataset

# Seed everything for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Imports OK")

Mounted at /content/drive
Device: cuda
Imports OK


In [ ]:
# Cell 2 — Config Block

CONFIG = {
    # Data
    "seq_len": 64,
    "batch_size": 64,

    # Model
    "hidden_size": 128,
    "embed_size": 128,
    "num_layers": 2,
    "dropout": 0.3,
    "nhead": 4,                  # Transformer: 128 / 4 = 32 per head ✓

    # Training
    "epochs": 10,
    "lr": 1e-3,
    "grad_clip": 1.0,
    "patience": 3,               # Early stopping patience

    # Paths
    "base_dir": "/content/drive/MyDrive/GP_NAS/baselines",
    "checkpoint_dir": "/content/drive/MyDrive/GP_NAS/baselines/checkpoints",
    "log_dir": "/content/drive/MyDrive/GP_NAS/baselines/logs",
    "results_path": "/content/drive/MyDrive/GP_NAS/baselines/baselines_results.json",
}

# Create all directories
for path in [CONFIG["base_dir"], CONFIG["checkpoint_dir"], CONFIG["log_dir"]]:
    os.makedirs(path, exist_ok=True)

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

print("\nDirectories created:")
for path in [CONFIG["base_dir"], CONFIG["checkpoint_dir"], CONFIG["log_dir"]]:
    print(f"  {path} — {'EXISTS' if os.path.exists(path) else 'FAILED'}")

Config:
  seq_len: 64
  batch_size: 64
  hidden_size: 128
  embed_size: 128
  num_layers: 2
  dropout: 0.3
  nhead: 4
  epochs: 10
  lr: 0.001
  grad_clip: 1.0
  patience: 3
  base_dir: /content/drive/MyDrive/GP_NAS/baselines
  checkpoint_dir: /content/drive/MyDrive/GP_NAS/baselines/checkpoints
  log_dir: /content/drive/MyDrive/GP_NAS/baselines/logs
  results_path: /content/drive/MyDrive/GP_NAS/baselines/baselines_results.json

Directories created:
  /content/drive/MyDrive/GP_NAS/baselines — EXISTS
  /content/drive/MyDrive/GP_NAS/baselines/checkpoints — EXISTS
  /content/drive/MyDrive/GP_NAS/baselines/logs — EXISTS


In [ ]:
# Cell 3 — Data Pipeline

# Load WikiText-2
print("Loading WikiText-2...")
raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Build vocabulary from train split only
print("Building vocabulary...")
from collections import Counter

def tokenize(text):
    return text.lower().split()

counter = Counter()
for line in raw_dataset["train"]["text"]:
    tokens = tokenize(line)
    counter.update(tokens)

# Keep tokens with frequency >= 2
MIN_FREQ = 2
special_tokens = ["<pad>", "<unk>"]
vocab_tokens = [tok for tok, freq in counter.items() if freq >= MIN_FREQ]
vocab = special_tokens + sorted(vocab_tokens)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

PAD_IDX = word2idx["<pad>"]
UNK_IDX = word2idx["<unk>"]
VOCAB_SIZE = len(vocab)

print(f"Vocabulary size: {VOCAB_SIZE}")

# Encode a list of lines into a flat token id tensor
def encode(lines):
    ids = []
    for line in lines:
        tokens = tokenize(line)
        ids.extend([word2idx.get(t, UNK_IDX) for t in tokens])
    return torch.tensor(ids, dtype=torch.long)

train_ids = encode(raw_dataset["train"]["text"])
val_ids   = encode(raw_dataset["validation"]["text"])
test_ids  = encode(raw_dataset["test"]["text"])

print(f"Train tokens : {len(train_ids):,}")
print(f"Val tokens   : {len(val_ids):,}")
print(f"Test tokens  : {len(test_ids):,}")

# Dataset — sliding window chunks of seq_len
class TextDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        self.seq_len = seq_len
        # Trim to exact multiple of seq_len
        n = (len(token_ids) - 1) // seq_len * seq_len
        self.data = token_ids[:n + 1]

    def __len__(self):
        return (len(self.data) - 1) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = self.data[start:start + self.seq_len]
        y = self.data[start + 1:start + self.seq_len + 1]
        return x, y

# Create datasets
train_dataset = TextDataset(train_ids, CONFIG["seq_len"])
val_dataset   = TextDataset(val_ids,   CONFIG["seq_len"])
test_dataset  = TextDataset(test_ids,  CONFIG["seq_len"])

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"], shuffle=False, drop_last=True)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")
print("\nData pipeline OK")

Loading WikiText-2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Building vocabulary...
Vocabulary size: 39275
Train tokens : 2,051,910
Val tokens   : 213,886
Test tokens  : 241,211

Train batches : 500
Val batches   : 52
Test batches  : 58

Data pipeline OK


In [ ]:
# Cell 4 — Model Definitions

import math

# ── 1. Plain GRU ────────────────────────────────────────────────
class PlainGRU(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.dropout(self.embedding(x))       # (B, T, E)
        out, hidden = self.gru(x, hidden)          # (B, T, H)
        out = self.dropout(out)
        logits = self.fc(out)                      # (B, T, V)
        return logits, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(
            CONFIG["num_layers"], batch_size, CONFIG["hidden_size"]
        ).to(device)


# ── 2. Plain LSTM ────────────────────────────────────────────────
class PlainLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.dropout(self.embedding(x))        # (B, T, E)
        out, hidden = self.lstm(x, hidden)          # (B, T, H)
        out = self.dropout(out)
        logits = self.fc(out)                       # (B, T, V)
        return logits, hidden

    def init_hidden(self, batch_size):
        h = torch.zeros(CONFIG["num_layers"], batch_size, CONFIG["hidden_size"]).to(device)
        c = torch.zeros(CONFIG["num_layers"], batch_size, CONFIG["hidden_size"]).to(device)
        return (h, c)


# ── 3. Plain Transformer ─────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, dropout, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, embed_size, 2).float() * (-math.log(10000.0) / embed_size)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)                        # (1, max_len, E)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class PlainTransformer(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers, nhead, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_IDX)
        self.pos_enc = PositionalEncoding(embed_size, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_size,
            nhead=nhead,
            dim_feedforward=hidden_size * 4,   # standard convention: 4x hidden
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_size, vocab_size)
        self.embed_size = embed_size

    def forward(self, x, hidden=None):
        seq_len = x.size(1)
        # Causal mask — prevents attending to future tokens
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(device)
        x = self.pos_enc(self.embedding(x) * math.sqrt(self.embed_size))  # (B, T, E)
        out = self.transformer(x, mask=causal_mask, is_causal=True)        # (B, T, E)
        logits = self.fc(out)                                               # (B, T, V)
        return logits, None   # None keeps the interface uniform with GRU/LSTM

    def init_hidden(self, batch_size):
        return None           # Transformer is stateless


# ── Instantiate all 3 ────────────────────────────────────────────
gru_model = PlainGRU(
    vocab_size=VOCAB_SIZE,
    embed_size=CONFIG["embed_size"],
    hidden_size=CONFIG["hidden_size"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"]
).to(device)

lstm_model = PlainLSTM(
    vocab_size=VOCAB_SIZE,
    embed_size=CONFIG["embed_size"],
    hidden_size=CONFIG["hidden_size"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"]
).to(device)

transformer_model = PlainTransformer(
    vocab_size=VOCAB_SIZE,
    embed_size=CONFIG["embed_size"],
    hidden_size=CONFIG["hidden_size"],
    num_layers=CONFIG["num_layers"],
    nhead=CONFIG["nhead"],
    dropout=CONFIG["dropout"]
).to(device)

# Parameter counts
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"PlainGRU         params: {count_params(gru_model):>10,}")
print(f"PlainLSTM        params: {count_params(lstm_model):>10,}")
print(f"PlainTransformer params: {count_params(transformer_model):>10,}")
print("\nModel definitions OK")

PlainGRU         params: 10,291,819
PlainLSTM        params: 10,357,867
PlainTransformer params: 10,490,219

Model definitions OK


In [ ]:
# Cell 5 — Metrics Utilities

import csv

# ── Perplexity ───────────────────────────────────────────────────
def perplexity_from_loss(loss):
    return math.exp(min(loss, 700))   # clamp to avoid overflow

# ── Top-K Accuracy ───────────────────────────────────────────────
def topk_accuracy(logits, targets, k, pad_idx=PAD_IDX, unk_idx=UNK_IDX):
    """
    logits  : (B, T, V)
    targets : (B, T)
    Excludes PAD and UNK positions from accuracy computation.
    """
    B, T, V = logits.size()
    logits  = logits.reshape(B * T, V)
    targets = targets.reshape(B * T)

    # Mask out PAD and UNK
    mask = (targets != pad_idx) & (targets != unk_idx)
    logits  = logits[mask]
    targets = targets[mask]

    if targets.numel() == 0:
        return 0.0

    _, top_k_preds = logits.topk(k, dim=-1)           # (N, k)
    correct = top_k_preds.eq(targets.unsqueeze(1))     # (N, k)
    return correct.any(dim=1).float().mean().item()


# ── Parameter Count ──────────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── Save Loss History to CSV ─────────────────────────────────────
def save_history_csv(history, model_name):
    path = os.path.join(CONFIG["log_dir"], f"{model_name}_history.csv")
    keys = ["epoch", "train_loss", "val_loss", "val_perplexity"]
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for row in history:
            writer.writerow({k: row[k] for k in keys})
    print(f"  History saved → {path}")


# ── Quick Sanity Check ───────────────────────────────────────────
dummy_logits  = torch.randn(4, 64, VOCAB_SIZE).to(device)
dummy_targets = torch.randint(0, VOCAB_SIZE, (4, 64)).to(device)

top1 = topk_accuracy(dummy_logits, dummy_targets, k=1)
top3 = topk_accuracy(dummy_logits, dummy_targets, k=3)
ppl  = perplexity_from_loss(5.0)

print(f"Top-1 accuracy (random logits, expect ~1/vocab): {top1:.6f}")
print(f"Top-3 accuracy (random logits, expect ~3/vocab): {top3:.6f}")
print(f"Perplexity at loss=5.0: {ppl:.2f}")
print("\nMetrics utilities OK")

Top-1 accuracy (random logits, expect ~1/vocab): 0.000000
Top-3 accuracy (random logits, expect ~3/vocab): 0.000000
Perplexity at loss=5.0: 148.41

Metrics utilities OK


In [ ]:
# Cell 6 — Generic Training Loop

def train_model(model, model_name, train_loader, val_loader):
    print(f"\n{'='*50}")
    print(f"  Training: {model_name}")
    print(f"{'='*50}")

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    best_val_loss = float("inf")
    patience_counter = 0
    history = []
    start_time = time.time()

    for epoch in range(1, CONFIG["epochs"] + 1):
        # ── Train Phase ───────────────────────────────────────────
        model.train()
        total_train_loss = 0
        total_train_batches = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)

            # Init hidden for RNNs, None for Transformer
            hidden = model.init_hidden(x.size(0))

            # Detach hidden state to prevent backprop through full history
            if isinstance(hidden, tuple):
                hidden = tuple(h.detach() for h in hidden)
            elif hidden is not None:
                hidden = hidden.detach()

            optimizer.zero_grad()
            logits, hidden = model(x, hidden)        # (B, T, V)

            # Reshape for CrossEntropyLoss: (B*T, V) vs (B*T,)
            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                y.reshape(-1)
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            optimizer.step()

            total_train_loss += loss.item()
            total_train_batches += 1

        avg_train_loss = total_train_loss / total_train_batches

        # ── Validation Phase ──────────────────────────────────────
        model.eval()
        total_val_loss = 0
        total_val_batches = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                hidden = model.init_hidden(x.size(0))
                logits, _ = model(x, hidden)
                loss = criterion(
                    logits.reshape(-1, VOCAB_SIZE),
                    y.reshape(-1)
                )
                total_val_loss += loss.item()
                total_val_batches += 1

        avg_val_loss = total_val_loss / total_val_batches
        val_ppl = perplexity_from_loss(avg_val_loss)

        # ── Logging ───────────────────────────────────────────────
        history.append({
            "epoch": epoch,
            "train_loss": round(avg_train_loss, 4),
            "val_loss": round(avg_val_loss, 4),
            "val_perplexity": round(val_ppl, 2)
        })

        print(f"  Epoch {epoch:02d}/{CONFIG['epochs']} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val PPL: {val_ppl:.2f}")

        # ── Save Best Checkpoint ──────────────────────────────────
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_path = os.path.join(CONFIG["checkpoint_dir"], f"{model_name}_best.pt")
            torch.save(model.state_dict(), best_path)
            print(f"    ✓ Best model saved (val_loss={best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"    No improvement. Patience: {patience_counter}/{CONFIG['patience']}")

        # ── Early Stopping ────────────────────────────────────────
        if patience_counter >= CONFIG["patience"]:
            print(f"\n  Early stopping triggered at epoch {epoch}")
            break

    # ── Save Final Checkpoint ─────────────────────────────────────
    final_path = os.path.join(CONFIG["checkpoint_dir"], f"{model_name}_final.pt")
    torch.save(model.state_dict(), final_path)

    total_time = time.time() - start_time
    print(f"\n  Training complete in {total_time/60:.2f} minutes")
    print(f"  Best Val Loss : {best_val_loss:.4f}")
    print(f"  Best Val PPL  : {perplexity_from_loss(best_val_loss):.2f}")

    # Save history CSV
    save_history_csv(history, model_name)

    return {
        "model_name": model_name,
        "best_val_loss": round(best_val_loss, 4),
        "best_val_ppl": round(perplexity_from_loss(best_val_loss), 2),
        "train_time_minutes": round(total_time / 60, 2),
        "param_count": count_params(model),
        "history": history
    }

print("Training loop defined OK")

Training loop defined OK


In [ ]:
# Config fix — update epochs to 20
CONFIG["epochs"] = 20
print("Epochs updated to:", CONFIG["epochs"])

Epochs updated to: 20


In [ ]:
# Cell 7 — Train All 3 Models Sequentially

all_results = {}

# ── 1. Train GRU ─────────────────────────────────────────────────
gru_result = train_model(
    model=gru_model,
    model_name="gru",
    train_loader=train_loader,
    val_loader=val_loader
)
all_results["gru"] = gru_result

# ── 2. Train LSTM ─────────────────────────────────────────────────
lstm_result = train_model(
    model=lstm_model,
    model_name="lstm",
    train_loader=train_loader,
    val_loader=val_loader
)
all_results["lstm"] = lstm_result

# ── 3. Train Transformer ──────────────────────────────────────────
transformer_result = train_model(
    model=transformer_model,
    model_name="transformer",
    train_loader=train_loader,
    val_loader=val_loader
)
all_results["transformer"] = transformer_result

# ── Quick Summary ─────────────────────────────────────────────────
print("\n" + "="*50)
print("  TRAINING SUMMARY")
print("="*50)
print(f"{'Model':<15} {'Best Val Loss':>14} {'Best Val PPL':>13} {'Time (min)':>11}")
print("-"*55)
for name, res in all_results.items():
    print(f"{name:<15} {res['best_val_loss']:>14.4f} "
          f"{res['best_val_ppl']:>13.2f} "
          f"{res['train_time_minutes']:>11.2f}")
print("="*55)


  Training: gru
  Epoch 01/20 | Train Loss: 6.2872 | Val Loss: 5.8840 | Val PPL: 359.24
    ✓ Best model saved (val_loss=5.8840)
  Epoch 02/20 | Train Loss: 6.0934 | Val Loss: 5.7677 | Val PPL: 319.80
    ✓ Best model saved (val_loss=5.7677)
  Epoch 03/20 | Train Loss: 5.9540 | Val Loss: 5.6809 | Val PPL: 293.23
    ✓ Best model saved (val_loss=5.6809)
  Epoch 04/20 | Train Loss: 5.8456 | Val Loss: 5.6188 | Val PPL: 275.55
    ✓ Best model saved (val_loss=5.6188)
  Epoch 05/20 | Train Loss: 5.7598 | Val Loss: 5.5764 | Val PPL: 264.12
    ✓ Best model saved (val_loss=5.5764)
  Epoch 06/20 | Train Loss: 5.6916 | Val Loss: 5.5439 | Val PPL: 255.67
    ✓ Best model saved (val_loss=5.5439)
  Epoch 07/20 | Train Loss: 5.6331 | Val Loss: 5.5138 | Val PPL: 248.09
    ✓ Best model saved (val_loss=5.5138)
  Epoch 08/20 | Train Loss: 5.5829 | Val Loss: 5.4938 | Val PPL: 243.17
    ✓ Best model saved (val_loss=5.4938)
  Epoch 09/20 | Train Loss: 5.5389 | Val Loss: 5.4728 | Val PPL: 238.12
    ✓ B

In [ ]:
# Recovery Cell — run this if runtime disconnected overnight

# Load best checkpoints from Drive
gru_model.load_state_dict(torch.load(
    os.path.join(CONFIG["checkpoint_dir"], "gru_best.pt"), map_location=device))

lstm_model.load_state_dict(torch.load(
    os.path.join(CONFIG["checkpoint_dir"], "lstm_best.pt"), map_location=device))

transformer_model.load_state_dict(torch.load(
    os.path.join(CONFIG["checkpoint_dir"], "transformer_best.pt"), map_location=device))

print("All 3 models loaded from Drive ✓")

All 3 models loaded from Drive ✓


In [ ]:
# Recovery Cell — rebuild all_results from last night's known values

all_results = {
    "gru": {
        "model_name"       : "gru",
        "best_val_loss"    : 5.3481,
        "best_val_ppl"     : 210.20,
        "train_time_minutes": 13.92,
        "param_count"      : count_params(gru_model),
    },
    "lstm": {
        "model_name"       : "lstm",
        "best_val_loss"    : 5.5195,
        "best_val_ppl"     : 249.51,
        "train_time_minutes": 14.09,
        "param_count"      : count_params(lstm_model),
    },
    "transformer": {
        "model_name"       : "transformer",
        "best_val_loss"    : 5.7420,
        "best_val_ppl"     : 311.70,
        "train_time_minutes": 14.03,
        "param_count"      : count_params(transformer_model),
    },
}

# Now merge test results
all_results["gru"].update(gru_test)
all_results["lstm"].update(lstm_test)
all_results["transformer"].update(transformer_test)

# Verify
print("all_results rebuilt successfully")
print("="*70)
print(f"{'Model':<15} {'Test Loss':>10} {'Test PPL':>10} {'Top-1 %':>10} {'Top-3 %':>10}")
print("-"*70)
for name, res in all_results.items():
    print(f"{name:<15} {res['test_loss']:>10.4f} "
          f"{res['test_perplexity']:>10.2f} "
          f"{res['top1_accuracy']*100:>10.2f} "
          f"{res['top3_accuracy']*100:>10.2f}")
print("="*70)

all_results rebuilt successfully
Model            Test Loss   Test PPL    Top-1 %    Top-3 %
----------------------------------------------------------------------
gru                 5.3102     202.39      21.57      33.59
lstm                5.4824     240.42      20.39      31.97
transformer         5.6811     293.29      18.42      30.29


In [ ]:
# Cell 8 — Test Evaluation

def evaluate_test(model, model_name, test_loader):
    print(f"\nEvaluating: {model_name}")

    # Load best checkpoint from Drive
    best_path = os.path.join(CONFIG["checkpoint_dir"], f"{model_name}_best.pt")
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    total_loss = 0
    total_batches = 0
    total_top1 = 0
    total_top3 = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            hidden = model.init_hidden(x.size(0))

            logits, _ = model(x, hidden)   # (B, T, V)

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                y.reshape(-1)
            )

            total_loss    += loss.item()
            total_top1    += topk_accuracy(logits, y, k=1)
            total_top3    += topk_accuracy(logits, y, k=3)
            total_batches += 1

    avg_loss = total_loss    / total_batches
    avg_top1 = total_top1   / total_batches
    avg_top3 = total_top3   / total_batches
    test_ppl = perplexity_from_loss(avg_loss)

    print(f"  Test Loss        : {avg_loss:.4f}")
    print(f"  Test Perplexity  : {test_ppl:.2f}")
    print(f"  Top-1 Accuracy   : {avg_top1*100:.2f}%")
    print(f"  Top-3 Accuracy   : {avg_top3*100:.2f}%")

    return {
        "test_loss"       : round(avg_loss,    4),
        "test_perplexity" : round(test_ppl,    2),
        "top1_accuracy"   : round(avg_top1,    4),
        "top3_accuracy"   : round(avg_top3,    4),
    }

# ── Evaluate all 3 ───────────────────────────────────────────────
gru_test         = evaluate_test(gru_model,         "gru",         test_loader)
lstm_test        = evaluate_test(lstm_model,         "lstm",        test_loader)
transformer_test = evaluate_test(transformer_model,  "transformer", test_loader)

# ── Merge into all_results ────────────────────────────────────────
all_results["gru"].update(gru_test)
all_results["lstm"].update(lstm_test)
all_results["transformer"].update(transformer_test)

# ── Print Summary ─────────────────────────────────────────────────
print("\n" + "="*70)
print("  TEST EVALUATION SUMMARY")
print("="*70)
print(f"{'Model':<15} {'Test Loss':>10} {'Test PPL':>10} {'Top-1 %':>10} {'Top-3 %':>10}")
print("-"*70)
for name, res in all_results.items():
    print(f"{name:<15} {res['test_loss']:>10.4f} "
          f"{res['test_perplexity']:>10.2f} "
          f"{res['top1_accuracy']*100:>10.2f} "
          f"{res['top3_accuracy']*100:>10.2f}")
print("="*70)


Evaluating: gru
  Test Loss        : 5.3102
  Test Perplexity  : 202.39
  Top-1 Accuracy   : 21.57%
  Top-3 Accuracy   : 33.59%

Evaluating: lstm
  Test Loss        : 5.4824
  Test Perplexity  : 240.42
  Top-1 Accuracy   : 20.39%
  Top-3 Accuracy   : 31.97%

Evaluating: transformer
  Test Loss        : 5.6811
  Test Perplexity  : 293.29
  Top-1 Accuracy   : 18.42%
  Top-3 Accuracy   : 30.29%

  TEST EVALUATION SUMMARY
Model            Test Loss   Test PPL    Top-1 %    Top-3 %
----------------------------------------------------------------------
gru                 5.3102     202.39      21.57      33.59
lstm                5.4824     240.42      20.39      31.97
transformer         5.6811     293.29      18.42      30.29


In [ ]:
# Cell 9 — Save All Results to Drive

import json

# ── Build final results dict ──────────────────────────────────────
final_results = {}

for name, res in all_results.items():
    final_results[name] = {
        "model_name"          : name,
        "param_count"         : res["param_count"],
        "train_time_minutes"  : res["train_time_minutes"],
        "best_val_loss"       : res["best_val_loss"],
        "best_val_perplexity" : res["best_val_ppl"],
        "test_loss"           : res["test_loss"],
        "test_perplexity"     : res["test_perplexity"],
        "top1_accuracy"       : res["top1_accuracy"],
        "top3_accuracy"       : res["top3_accuracy"],
    }

# ── Save baselines_results.json ───────────────────────────────────
results_path = CONFIG["results_path"]
with open(results_path, "w") as f:
    json.dump(final_results, f, indent=4)

print(f"Results saved → {results_path}")

# ── Verify file exists and is readable ───────────────────────────
with open(results_path, "r") as f:
    verify = json.load(f)

print("\nVerification — loaded back from Drive:")
print(json.dumps(verify, indent=4))

Results saved → /content/drive/MyDrive/GP_NAS/baselines/baselines_results.json

Verification — loaded back from Drive:
{
    "gru": {
        "model_name": "gru",
        "param_count": 10291819,
        "train_time_minutes": 13.92,
        "best_val_loss": 5.3481,
        "best_val_perplexity": 210.2,
        "test_loss": 5.3102,
        "test_perplexity": 202.39,
        "top1_accuracy": 0.2157,
        "top3_accuracy": 0.3359
    },
    "lstm": {
        "model_name": "lstm",
        "param_count": 10357867,
        "train_time_minutes": 14.09,
        "best_val_loss": 5.5195,
        "best_val_perplexity": 249.51,
        "test_loss": 5.4824,
        "test_perplexity": 240.42,
        "top1_accuracy": 0.2039,
        "top3_accuracy": 0.3197
    },
    "transformer": {
        "model_name": "transformer",
        "param_count": 10490219,
        "train_time_minutes": 14.03,
        "best_val_loss": 5.742,
        "best_val_perplexity": 311.7,
        "test_loss": 5.6811,
        "te

In [ ]:
# Cell 10 — Final Comparison Table

# ── Load from Drive (safe — works even after reconnect) ───────────
with open(CONFIG["results_path"], "r") as f:
    final_results = json.load(f)

# ── Print Comparison Table ────────────────────────────────────────
print("\n" + "="*85)
print("  BASELINE MODELS — FINAL COMPARISON TABLE")
print("  Dataset: WikiText-2 | Seq Len: 64 | Batch: 64 | Hidden: 128 | Epochs: 20 (max)")
print("="*85)

header = (f"{'Model':<15} {'Params':>10} {'Val PPL':>9} {'Test PPL':>10} "
          f"{'Top-1 %':>9} {'Top-3 %':>9} {'Time(min)':>10}")
print(header)
print("-"*85)

for name, res in final_results.items():
    print(f"{name:<15} "
          f"{res['param_count']:>10,} "
          f"{res['best_val_perplexity']:>9.2f} "
          f"{res['test_perplexity']:>10.2f} "
          f"{res['top1_accuracy']*100:>9.2f} "
          f"{res['top3_accuracy']*100:>9.2f} "
          f"{res['train_time_minutes']:>10.2f}")

print("="*85)

# ── Best model per metric ─────────────────────────────────────────
print("\n  BEST PER METRIC:")
print(f"  Lowest Test PPL  → {min(final_results, key=lambda x: final_results[x]['test_perplexity'])}")
print(f"  Highest Top-1 %  → {max(final_results, key=lambda x: final_results[x]['top1_accuracy'])}")
print(f"  Highest Top-3 %  → {max(final_results, key=lambda x: final_results[x]['top3_accuracy'])}")
print(f"  Fastest Training → {min(final_results, key=lambda x: final_results[x]['train_time_minutes'])}")
print(f"  Fewest Params    → {min(final_results, key=lambda x: final_results[x]['param_count'])}")

# ── Drive contents summary ────────────────────────────────────────
print("\n" + "="*85)
print("  FILES SAVED TO DRIVE")
print("="*85)
for root, dirs, files in os.walk(CONFIG["base_dir"]):
    for file in files:
        fpath = os.path.join(root, file)
        fsize = os.path.getsize(fpath) / (1024*1024)
        print(f"  {fpath}  ({fsize:.2f} MB)")

print("\nBaseline notebook complete ✓")


  BASELINE MODELS — FINAL COMPARISON TABLE
  Dataset: WikiText-2 | Seq Len: 64 | Batch: 64 | Hidden: 128 | Epochs: 20 (max)
Model               Params   Val PPL   Test PPL   Top-1 %   Top-3 %  Time(min)
-------------------------------------------------------------------------------------
gru             10,291,819    210.20     202.39     21.57     33.59      13.92
lstm            10,357,867    249.51     240.42     20.39     31.97      14.09
transformer     10,490,219    311.70     293.29     18.42     30.29      14.03

  BEST PER METRIC:
  Lowest Test PPL  → gru
  Highest Top-1 %  → gru
  Highest Top-3 %  → gru
  Fastest Training → gru
  Fewest Params    → gru

  FILES SAVED TO DRIVE
  /content/drive/MyDrive/GP_NAS/baselines/baselines_results.json  (0.00 MB)
  /content/drive/MyDrive/GP_NAS/baselines/checkpoints/gru_final.pt  (39.26 MB)
  /content/drive/MyDrive/GP_NAS/baselines/checkpoints/gru_best.pt  (39.26 MB)
  /content/drive/MyDrive/GP_NAS/baselines/checkpoints/lstm_final.pt  (3